# Generate Postcode DC Scope

Builds a Data Correctness (DC) scoping CSV that sets `addr:postcode` on APTs, from a
MNR-vs-Orbis comparison CSV (columns `orbis_id`, `mnr_postal_code`, street / house number /
place match details, geometry WKT, distance, ...).

## What this notebook does

1. **Read the input CSV** with pandas (`orbis_id` and `mnr_postal_code` are read as text so
   long ids and leading-zero postcodes survive)
2. **Drop rows** that have no `orbis_id` or no `mnr_postal_code` — every other row is kept as-is
3. **Map columns** to the DC scope format and write the output CSV
4. **Validate** the output: row count, column order, constant values, no blanks in the required fields

## Column mapping

| Output column | Source |
|---|---|
| `Product_orbis_id` | `orbis_id` |
| `Target_Value` | `mnr_postal_code` |
| `Product_Address_component` | constant `addr:postcode` |
| `Product_Language` | constant `es-Latn` |
| `Source_Feature_Id` | blank |
| `Lat` | constant `0` |
| `Lon` | constant `0` |

### Load input CSV

In [ ]:
%pip install -r requirements.txt

In [ ]:
import pandas as pd

INPUT_CSV = "/Users/sawan.darekar/Downloads/postal_code_comparison.csv"  # <-- update this path

# keep both key columns as text: orbis_id is a long composite id, postcodes can have leading zeros
input_df = pd.read_csv(INPUT_CSV, dtype={"orbis_id": str, "mnr_postal_code": str})

print(f"Input rows    : {len(input_df)}")
print(f"Input columns : {list(input_df.columns)}")
display(input_df.head())

### Generate DC scope CSV

In [ ]:
OUTPUT_CSV = "/Users/sawan.darekar/Downloads/postcode-dc-scope.csv"  # <-- update this path

ADDRESS_COMPONENT = "addr:postcode"
LANGUAGE = "es-Latn"

SCOPE_COLUMNS = [
    "Product_orbis_id",
    "Target_Value",
    "Product_Address_component",
    "Product_Language",
    "Source_Feature_Id",
    "Lat",
    "Lon",
]

work_df = input_df[["orbis_id", "mnr_postal_code"]].copy()
work_df["orbis_id"] = work_df["orbis_id"].astype("string").str.strip()
work_df["mnr_postal_code"] = work_df["mnr_postal_code"].astype("string").str.strip()

# treat empty strings as missing, then drop the incomplete rows (no other filtering)
work_df = work_df.replace("", pd.NA)
dropped_df = work_df[work_df["orbis_id"].isna() | work_df["mnr_postal_code"].isna()]
work_df = work_df.drop(dropped_df.index)

scope_df = pd.DataFrame({
    "Product_orbis_id": work_df["orbis_id"].to_numpy(),
    "Target_Value": work_df["mnr_postal_code"].to_numpy(),
    "Product_Address_component": ADDRESS_COMPONENT,
    "Product_Language": LANGUAGE,
    "Source_Feature_Id": "",
    "Lat": 0,
    "Lon": 0,
})[SCOPE_COLUMNS]

scope_df.to_csv(OUTPUT_CSV, index=False)

In [ ]:
print(f"Input rows                      : {len(input_df)}")
print(f"Dropped (missing id / postcode) : {len(dropped_df)}")
print(f"Scope rows written              : {len(scope_df)}")
print(f"Unique orbis ids                : {scope_df['Product_orbis_id'].nunique()}")
print(f"Output file                     : {OUTPUT_CSV}")
display(scope_df.head(10))

### Validate output CSV

In [ ]:
check_df = pd.read_csv(OUTPUT_CSV, dtype=str, keep_default_na=False)

checks = {
    "column order matches expected": list(check_df.columns) == SCOPE_COLUMNS,
    "row count matches input minus dropped": len(check_df) == len(input_df) - len(dropped_df),
    "no blank Product_orbis_id": (check_df["Product_orbis_id"].str.strip() != "").all(),
    "no blank Target_Value": (check_df["Target_Value"].str.strip() != "").all(),
    "Product_Address_component is constant": set(check_df["Product_Address_component"]) == {ADDRESS_COMPONENT},
    "Product_Language is constant": set(check_df["Product_Language"]) == {LANGUAGE},
    "Source_Feature_Id is blank": (check_df["Source_Feature_Id"] == "").all(),
    "Lat is 0": (check_df["Lat"] == "0").all(),
    "Lon is 0": (check_df["Lon"] == "0").all(),
}

for name, passed in checks.items():
    print(f"  {'PASS' if passed else 'FAIL'}  {name}")

print()
print("All checks passed." if all(checks.values()) else "WARNING: some checks failed!")